# 📖 Module 5: LangChain Essentials - Templates, Chains, & Tools (Explanation)

Welcome to **Module 5**! In this comprehensive masterclass, you will learn the core building blocks of modern LLM applications using **LangChain**:
1. 📝 **Prompt Templates**: Parameterize and structure system/user prompts cleanly (`PromptTemplate` & `ChatPromptTemplate`).
2. 🔗 **Chains & LCEL**: Compose production-ready multi-stage workflows using **LangChain Expression Language** (`|` pipe operator).
3. 🛠️ **Tools & Function Calling**: Supercharge LLMs by binding external Python functions/tools (`@tool`, `bind_tools()`, and `ToolMessage`).

We will power our workflow using **OpenRouter** with the **Cohere North Mini Code (`cohere/north-mini-code:free`)** model!

---
> 🎓 **INSTRUCTOR NOTES & TEACHING GUIDE**
> 
> **Lesson Target:** ~60 minutes
> 
> **Key Teaching Objectives:**
> 1. Demonstrate model initialization using `ChatOpenAI` pointing to OpenRouter (`https://openrouter.ai/api/v1`).
> 2. Differentiate between single-string `PromptTemplate` and multi-turn `ChatPromptTemplate`.
> 3. Master LCEL pipe syntax `prompt | llm | StrOutputParser()`, sequential chains, and `JsonOutputParser`.
> 4. Teach native tool calling: defining tools with `@tool`, binding via `.bind_tools()`, inspecting `AIMessage.tool_calls`, and completing the execution loop with `ToolMessage`.
> 
> **Common Student Pitfalls:**
> - *Missing `openai_api_base`:* When using OpenRouter with `ChatOpenAI`, students must set `openai_api_base="https://openrouter.ai/api/v1"`.
> - *Tool Docstrings:* The LLM uses function docstrings and type annotations to decide when and how to call tools. Remind students to write descriptive docstrings!


## 1️⃣ Initializing the LLM via OpenRouter (`cohere/north-mini-code:free`)

LangChain provides the `ChatOpenAI` class which can seamlessly interface with any OpenAI-compatible API host like **OpenRouter**.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

OPENROUTER_API_KEY = "OPENROUTER_API_KEY"
MODEL_NAME = "cohere/north-mini-code:free"

# Step 1: Initialize ChatOpenAI pointing to OpenRouter base URL
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.2
)

# Step 2: Test basic call with chat messages
messages = [
    SystemMessage(content="You are an expert AI software architect."),
    HumanMessage(content="Explain what LangChain is in two concise sentences.")
]

response = llm.invoke(messages)
print("--- Direct LLM Response ---")
print(response.content)


--- Direct LLM Response ---
LangChain is an open‑source framework for building applications that combine large language models with external data sources and tools, enabling developers to chain together model calls, memory, prompts, and actions in a modular workflow. It abstracts common patterns—such as prompt templates, conversation memory, and tool integration—so you can rapidly prototype and deploy AI agents, chatbots, and reasoning systems that go beyond simple completions.


## 2️⃣ Parameterized Prompts (`PromptTemplate` & `ChatPromptTemplate`)

Hardcoding variables into raw strings leads to messy code and formatting bugs. LangChain provides templates to define reusable input placeholders.


In [2]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# 2.1 String PromptTemplate
string_prompt = PromptTemplate.from_template(
    "Explain the concept of {concept} to a {audience} in {num_sentences} concise sentences."
)
formatted_str = string_prompt.format(concept="Recursion", audience="10-year-old child", num_sentences=2)
print("--- Formatted String Prompt ---")
print(formatted_str)

# 2.2 ChatPromptTemplate (System + Human message roles)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class software engineer specializing in {domain}."),
    ("human", "Provide 3 best practices for designing scalable {technology} architectures.")
])

formatted_chat = chat_prompt.format_messages(domain="Cloud Infrastructure", technology="Microservices")
print("\n--- Formatted Chat Prompt ---")
for msg in formatted_chat:
    print(f"[{msg.type.upper()}]: {msg.content}")


--- Formatted String Prompt ---
Explain the concept of Recursion to a 10-year-old child in 2 concise sentences.

--- Formatted Chat Prompt ---
[SYSTEM]: You are a world-class software engineer specializing in Cloud Infrastructure.
[HUMAN]: Provide 3 best practices for designing scalable Microservices architectures.


## 3️⃣ Chains & LangChain Expression Language (LCEL)

**LCEL** allows you to build composite chains using the Unix-style pipe operator (`|`). 
An LCEL chain naturally streams inputs, outputs, and standardizes component execution: `Prompt | Model | Parser`.


In [3]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# 3.1 Simple LCEL Chain with StrOutputParser
parser = StrOutputParser()
simple_chain = chat_prompt | llm | parser

print("--- 3.1 LCEL Chain Result ---")
res1 = simple_chain.invoke({"domain": "Backend Systems", "technology": "RESTful APIs"})
print(res1)

# 3.2 Sequential Multi-Stage LCEL Chain (Outline -> Code Implementation)
outline_prompt = PromptTemplate.from_template("Create a 3-step high-level plan to implement: {feature}")
outline_chain = outline_prompt | llm | parser

code_prompt = PromptTemplate.from_template("Based on this plan:\n{plan}\n\nWrite Python code implementing step 1:")
sequential_chain = {"plan": outline_chain} | code_prompt | llm | parser

print("\n--- 3.2 Sequential Chain Code Output ---")
res2 = sequential_chain.invoke({"feature": "JWT Authentication Middleware in FastAPI"})
print(res2)

# 3.3 Structured JSON Output Parsing
json_prompt = PromptTemplate.from_template(
    "Analyze the sentiment of this text: '{text}'.\n"
    "Respond strictly with a JSON object containing keys: 'sentiment' (positive/negative/neutral) and 'confidence' (float 0-1).\n"
    "{format_instructions}"
)
json_parser = JsonOutputParser()
json_prompt_partial = json_prompt.partial(format_instructions=json_parser.get_format_instructions())

json_chain = json_prompt_partial | llm | json_parser
parsed_output = json_chain.invoke({"text": "This framework is amazingly fast, clear, and elegant to use!"})

print("\n--- 3.3 Parsed JSON Output ---")
print("Parsed Object Type:", type(parsed_output))
print("Sentiment Value:", parsed_output.get("sentiment"))
print("Full JSON:", parsed_output)


--- 3.1 LCEL Chain Result ---


Below are three high‑impact best‑practices that will keep a RESTful API both **clean** and **ready to grow** as traffic, data volume, and client‑base expand.

---

## 1️⃣ Design Resources First – Use Proper URI Conventions & Hypermedia Controls  

| What to do | Why it matters for scalability |
|------------|--------------------------------|
| **Model the API around business entities** (e.g., `/users/{id}`, `/orders/{id}`) rather than actions or verbs. | Clients can discover resources, cache them independently, and the service can be sharded or cached per‑resource without breaking callers. |
| **Follow the “single‑resource‑per‑URI” rule** – one URI maps to one logical resource (collection or item). | Keeps the URI space predictable, simplifies routing, and enables granular rate‑limiting or caching per resource. |
| **Leverage HATEOAS (Hypermedia as the Engine of Application State)** – embed links (`self`, `next`, `create`) in responses. | Clients stay loosely coupled; when you need to 

Here is the Python code implementing **Step 1: Project Setup & Dependency Injection**.

This step is broken down into three essential components: the dependency list, the security configuration, and the data models.

### 1. Dependency Management (`requirements.txt`)
First, define the packages your project will use. Create a file named `requirements.txt` in your project root:

```text
fastapi
uvicorn[standard]
python-jose[cryptography]
passlib[bcrypt]
sqlalchemy
pydantic
python-multipart  # Useful for file uploads if needed later
```

### 2. Security Configuration (`app/core/config.py`)
Create a configuration module to manage your secret keys and security settings. This prevents hardcoding sensitive data directly into your source code.

```python
import os
from typing import Optional

class Settings:
    # Application Settings
    PROJECT_NAME: str = "FastAPI JWT Auth"
    VERSION: str = "1.0.0"
    
    # Security Settings
    # In production, always use environment variables for secre


--- 3.3 Parsed JSON Output ---
Parsed Object Type: <class 'dict'>
Sentiment Value: positive
Full JSON: {'sentiment': 'positive', 'confidence': 0.95}


## 4️⃣ Tools & Function Calling in LangChain

**Tools** allow LLMs to take actions or fetch real-time information (e.g. databases, math engines, APIs).
LangChain makes it easy to define tools with `@tool`, bind them to models via `.bind_tools()`, and execute them.


In [4]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

# 4.1 Define custom tools using @tool decorator
@tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers 'a' and 'b' and returns the product."""
    return a * b

@tool
def fetch_user_data(user_id: int) -> dict:
    """Fetches user profile information from database given a numerical user_id."""
    mock_db = {
        101: {"name": "Alice Vance", "role": "Lead Data Scientist", "status": "Active"},
        102: {"name": "Bob Smith", "role": "DevOps Architect", "status": "On Leave"}
    }
    return mock_db.get(user_id, {"error": "User ID not found"})

tools = [multiply, fetch_user_data]
print("Registered Tools:", [t.name for t in tools])

# 4.2 Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# 4.3 Invoke LLM with a query requiring tool selection
user_query = "Can you fetch the profile for user ID 101 and multiply 15.5 by 4?"
ai_msg = llm_with_tools.invoke(user_query)

print("\n--- Tool Calls Requested by Model ---")
for tc in ai_msg.tool_calls:
    print(f"Tool Name: {tc['name']}, Arguments: {tc['args']}, Call ID: {tc['id']}")

# 4.4 Complete the Execution Loop (Run tools & return ToolMessages to LLM)
tool_map = {t.name: t for t in tools}
conversation = [HumanMessage(content=user_query), ai_msg]

for tc in ai_msg.tool_calls:
    tool_func = tool_map[tc["name"]]
    tool_output = tool_func.invoke(tc["args"])
    print(f"Executed '{tc['name']}' -> Output: {tool_output}")
    conversation.append(ToolMessage(content=str(tool_output), tool_call_id=tc["id"]))

# Synthesize final natural language answer
final_answer = llm_with_tools.invoke(conversation)
print("\n--- Final Synthesized Response ---")
print(final_answer.content)


Registered Tools: ['multiply', 'fetch_user_data']



--- Tool Calls Requested by Model ---
Tool Name: fetch_user_data, Arguments: {'user_id': 101}, Call ID: fetch_user_data_tgqgw1swz2xw
Tool Name: multiply, Arguments: {'a': 15.5, 'b': 4}, Call ID: multiply_95f80663txdv
Executed 'fetch_user_data' -> Output: {'name': 'Alice Vance', 'role': 'Lead Data Scientist', 'status': 'Active'}
Executed 'multiply' -> Output: 62.0



--- Final Synthesized Response ---
Here are the results:

**User Profile for ID 101:**
- Name: Alice Vance
- Role: Lead Data Scientist
- Status: Active

**Multiplication Result:**
15.5 × 4 = 62.0
